##CAPSTON PROJECT - Bank marketing response ML

In [34]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report

import joblib

#Load the Dataset

In [35]:
df = pd.read_csv("/content/bank.csv")
y = df['deposit']
X = df.drop('deposit', axis=1)

In [36]:
df.isnull().sum()

,0
age,0
job,0
marital,0
education,0
default,0
balance,0
housing,0
loan,0
contact,0
day,0


In [37]:
df.dropna()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11157,33,blue-collar,single,primary,no,1,yes,no,cellular,20,apr,257,1,-1,0,unknown,no
11158,39,services,married,secondary,no,733,no,no,unknown,16,jun,83,4,-1,0,unknown,no
11159,32,technician,single,secondary,no,29,no,no,cellular,19,aug,156,2,-1,0,unknown,no
11160,43,technician,married,secondary,no,0,no,yes,cellular,8,may,9,2,172,5,failure,no


# Convert target variable to numeric


In [38]:
df['deposit'] = (df['deposit'] == 'yes').astype(int)

In [39]:
print("Class Distribution:")
print(df['deposit'].value_counts())

print("\nPercentage Distribution:")
print(df['deposit'].value_counts(normalize=True))

Class Distribution:
deposit
0    5873
1    5289
Name: count, dtype: int64

Percentage Distribution:
deposit
0    0.52616
1    0.47384
Name: proportion, dtype: float64


In [40]:
X=df.drop('deposit',axis=1)
y=df['deposit']

#Train_Test_Split

In [41]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y )

#Scaling the Columns

In [42]:
categorical_cols=X.select_dtypes(include=['object']).columns
numeric_cols=X.select_dtypes(exclude=['object']).columns
preprocessor=ColumnTransformer(
    transformers=[
        ('cat',OneHotEncoder(handle_unknown='ignore'),categorical_cols),
        ('num',StandardScaler(),numeric_cols)
    ]
)

#Model_Training

1.Logistic-Regression

In [43]:
lr_pipeline=Pipeline([
    ('preprocessor',preprocessor),
    ('model',LogisticRegression(max_iter=1000))
])
lr_pipeline.fit(X_train,y_train)
lr_pred=lr_pipeline.predict(X_test)

print("Logistic Regression Accuracy:",accuracy_score(y_test, lr_pred))
print(classification_report(y_test,lr_pred))

Logistic Regression Accuracy: 0.8262427227944469
              precision    recall  f1-score   support

           0       0.82      0.85      0.84      1175
           1       0.83      0.80      0.81      1058

    accuracy                           0.83      2233
   macro avg       0.83      0.82      0.83      2233
weighted avg       0.83      0.83      0.83      2233



#2.KNN

In [44]:
knn_pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('model',KNeighborsClassifier(n_neighbors=5))
])
knn_pipeline.fit(X_train,y_train)
knn_pred=knn_pipeline.predict(X_test)

print("KNN Accuracy:",accuracy_score(y_test,knn_pred))
print(classification_report(y_test,knn_pred))

KNN Accuracy: 0.8172861621137483
              precision    recall  f1-score   support

           0       0.82      0.84      0.83      1175
           1       0.82      0.79      0.80      1058

    accuracy                           0.82      2233
   macro avg       0.82      0.82      0.82      2233
weighted avg       0.82      0.82      0.82      2233



#3.RandomForest

In [45]:
rf_pipeline=Pipeline([
    ('preprocessor',preprocessor),
    ('model',RandomForestClassifier(
        n_estimators=100,
        max_depth=8,
        random_state=42
    ))
])
rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)
print("Random Forest Accuracy:",accuracy_score(y_test, rf_pred))
print(classification_report(y_test,rf_pred))

Random Forest Accuracy: 0.8450515002239141
              precision    recall  f1-score   support

           0       0.87      0.83      0.85      1175
           1       0.82      0.87      0.84      1058

    accuracy                           0.85      2233
   macro avg       0.85      0.85      0.84      2233
weighted avg       0.85      0.85      0.85      2233



#4.Navie_Bayes

In [46]:
from sklearn.naive_bayes import GaussianNB
nb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GaussianNB())
])
nb_pipeline.fit(X_train, y_train)
nb_pred = nb_pipeline.predict(X_test)
from sklearn.metrics import accuracy_score, classification_report
print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_pred))
print("\nClassification Report:\n", classification_report(y_test, nb_pred))

Naive Bayes Accuracy: 0.7201074787281684

Classification Report:
               precision    recall  f1-score   support

           0       0.69      0.86      0.76      1175
           1       0.78      0.57      0.66      1058

    accuracy                           0.72      2233
   macro avg       0.74      0.71      0.71      2233
weighted avg       0.73      0.72      0.71      2233



#Model_Comparison

In [47]:
results = {
    "Logistic Regression": accuracy_score(y_test, lr_pred),
    "KNN": accuracy_score(y_test, knn_pred),
    "Random Forest": accuracy_score(y_test, rf_pred),
    "Naive Bayes": accuracy_score(y_test, nb_pred)
}
print("\nModel Comparison:")
for model, acc in results.items():
    print(f"{model}: {acc:.4f}")


Model Comparison:
Logistic Regression: 0.8262
KNN: 0.8173
Random Forest: 0.8451
Naive Bayes: 0.7201


In [55]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_score
y_pred = rf_pipeline.predict(X_test)
y_prob = rf_pipeline.predict_proba(X_test)[:, 1]
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Train Accuracy:", rf_pipeline.score(X_train, y_train))
print("Test Accuracy:", rf_pipeline.score(X_test, y_test))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nROC-AUC Score:", roc_auc_score(y_test, y_prob))
cv_scores = cross_val_score(rf_pipeline, X, y, cv=5)
print("\nCross Validation Scores:", cv_scores)
print("Mean CV Score:", cv_scores.mean())

Test Accuracy: 0.8450515002239141
Train Accuracy: 0.864934483144809
Test Accuracy: 0.8450515002239141

Confusion Matrix:
 [[971 204]
 [142 916]]

Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.83      0.85      1175
           1       0.82      0.87      0.84      1058

    accuracy                           0.85      2233
   macro avg       0.85      0.85      0.84      2233
weighted avg       0.85      0.85      0.85      2233


ROC-AUC Score: 0.9122977918996098

Cross Validation Scores: [0.62427228 0.73130318 0.77956989 0.84453405 0.83154122]
Mean CV Score: 0.7622441240628115


Among all the models, Random Forest performed the best with the highest accuracy of 84.51%.

#Model_Saving

In [48]:
import joblib
joblib.dump(rf_pipeline,"bank_rf_model.pkl")
print("Model saved successfully!")

Model saved successfully!


In [54]:
loaded_model=joblib.load("bank_rf_model.pkl")
sample_data=pd.DataFrame([{
    'age': 35,
    'job': 'management',
    'marital':'married',
    'education':'tertiary',
    'default':'no',
    'balance':2000,
    'housing':'yes',
    'loan':'no',
    'contact':'cellular',
    'day':10,
    'month':'may',
    'duration':300,
    'campaign':1,
    'pdays':-1,
    'previous':0,
    'poutcome':'unknown'
}])
prediction=loaded_model.predict(sample_data)

print("Prediction:", prediction)
if prediction[0] == 1:
    print("Customer will SUBSCRIBE (Yes)")
else:
    print("Customer will NOT SUBSCRIBE (No)")


Prediction: [0]
Customer will NOT SUBSCRIBE (No)
